# Exploratory Data Analysis (filtered dataset)

## 1. Import libraries

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import plotly.express.colors as pc
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 2. Load metadata

In [2]:
df_all = pd.read_csv("metadata.csv")
df = pd.read_csv("fragments_metadata_filtered.csv")
df

,id,name,age,gender,position,record_id,segment,label,category,duration
0,1,P1,4.3,1,p4,7545,0,Normal,Normal,1.57725
1,2,P1,4.3,1,p4,7545,1,Rhonchi,Adventitious,0.95725
2,3,P1,4.3,1,p4,7545,2,Normal,Normal,1.01225
3,4,P2,5.3,0,p1,25271,0,Normal,Normal,2.12525
4,9,P3,4.3,1,p1,24116,0,Normal,Normal,2.40825
...,...,...,...,...,...,...,...,...,...,...
20866,24574,P957,8.4,0,p8,32670,3,Normal,Normal,0.94025
20867,24575,P957,8.4,0,p8,32670,4,Normal,Normal,0.99525
20868,24576,P957,8.4,0,p8,32670,5,Normal,Normal,0.68525
20869,24577,P957,8.4,0,p8,32670,6,Normal,Normal,1.19925


## 3. EDA

### 3.1. Dataset analysis

In [3]:
n_records = len(df_all)
n_poor_quality = sum(df_all["Poor Quality"])
n_segments = len(df)
n_patients = len(df["name"].unique())

tot = sum(df_all["duration"])
mins_t, s_t = int(tot//60), round(tot%60, 2)
h_t, mins_t = mins_t//60, mins_t%60

dur = sum(df["duration"])
mins, s = int(dur//60), round(dur%60, 2)
h, mins = mins//60, mins%60


print(f"Number of segments: {n_segments} ({n_segments/24578*100:.2f}%)")
print(f"Number of patients: {n_patients} ({n_patients/958*100:.2f}%)")
print(f"Corpus duration: {h} h {mins} min {s} s ({dur/42759.24*100:.2f}%)")

Number of segments: 20871 (84.92%)
Number of patients: 920 (96.03%)
Corpus duration: 10 h 15 min 7.04 s (86.31%)


### 3.2. Label distribution

In [4]:
# --- 1. Preparar datos ---
total_n = len(df)

counts = df.groupby(["category", "label"]).size().reset_index(name="count")
counts["proportion"] = counts["count"] / total_n

adv_counts = (
    df[df["label"] != "Normal"]["label"]
    .value_counts(normalize=True)
    .rename_axis("label")
    .reset_index(name="proportion")
)

adv_order = adv_counts.sort_values("proportion", ascending=False)["label"].tolist()
labels_order = ["Normal"] + adv_order

# --- 2. Paleta temática: verde-azulado para Normal, gradiente cálido (rojo->amarillo claro)
#     para adventicios, del más frecuente (más intenso) al menos frecuente (más claro) ---
n_adv = len(adv_order)
adv_colors = pc.sample_colorscale(
    "OrRd", [1 - 0.8 * (i / max(n_adv - 1, 1)) for i in range(n_adv)]
)
color_map = {"Normal": "#2CA58D"}
color_map.update({lab: c for lab, c in zip(adv_order, adv_colors)})

# --- 3. Crear figura ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Normal vs Adventitious", "Adventitious type distribution"),
    specs=[[{"type": "bar"}, {"type": "domain"}]],
)

# --- Subplot 1: barra apilada. Solo mostramos texto si el segmento es >= 4% ---
TEXT_THRESHOLD = 0.04
for lab in labels_order:
    sub = counts[counts["label"] == lab]
    if sub.empty:
        continue
    texts = sub["proportion"].map(lambda p: f"{p:.1%}" if p >= TEXT_THRESHOLD else "")
    fig.add_trace(
        go.Bar(
            x=sub["category"],
            y=sub["proportion"],
            name=lab,
            marker_color=color_map[lab],
            legendgroup=lab,
            text=texts,
            textposition="inside",
            textfont=dict(size=13, color="white"),
            hovertemplate=f"{lab}: %{{y:.1%}}<extra></extra>",
        ),
        row=1, col=1
    )

# --- Subplot 2: pie, sin labels, solo % con posicionamiento automático (evita solapes) ---
fig.add_trace(
    go.Pie(
        labels=adv_counts["label"],
        values=adv_counts["proportion"],
        name="Adventitious",
        marker_colors=[color_map[lab] for lab in adv_counts["label"]],
        textinfo="percent",
        textposition="auto",
        showlegend=False,
        sort=False,  # respeta el orden de más a menos que ya trae adv_counts... 
    ),
    row=1, col=2
)
# aseguramos orden más->menos en el pie también
fig.data[-1].labels = adv_order
fig.data[-1].values = [adv_counts.set_index("label").loc[lab, "proportion"] for lab in adv_order]
fig.data[-1].marker.colors = [color_map[lab] for lab in adv_order]

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    barmode="stack",
    title_text="Label distribution",
    showlegend=True,
    legend=dict(traceorder="normal", font=dict(size=14)),
    font=dict(size=15),
    title_font=dict(size=20),
    plot_bgcolor="white",
)
fig.update_annotations(font_size=16)

fig.update_xaxes(
    title_text="Category",
    categoryorder="array",
    categoryarray=["Normal", "Adventitious"],
    row=1, col=1
)
fig.update_yaxes(title_text="Proportion", tickformat=".0%", row=1, col=1)

fig.show()
fig.write_image("assets/filtered/label_distribution.png", scale=5)

### 3.3. Age distribution

In [5]:
# --- 1. Datos ---
ages = df_all.groupby(["patient_id", "age"])["age"].mean().dropna()

# --- 2. Estimar KDE ---
kde = gaussian_kde(ages)
x_kde = np.linspace(ages.min(), ages.max(), 300)
y_kde = kde(x_kde)

# --- 3. Crear figura: 2 filas, eje X compartido ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],   # histograma más grande, boxplot como "resumen" fino
    vertical_spacing=0.03,
)

# --- Histograma (normalizado a densidad para que la escala coincida con la KDE) ---
fig.add_trace(
    go.Histogram(
        x=ages,
        histnorm="probability density",
        name="Duration",
        marker_color="steelblue",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

# --- KDE ---
fig.add_trace(
    go.Scatter(
        x=x_kde,
        y=y_kde,
        mode="lines",
        name="KDE",
        line=dict(color="cadetblue", width=2.5),
        showlegend=False,
    ),
    row=1, col=1
)

# --- Boxplot horizontal ---
fig.add_trace(
    go.Box(
        x=ages,
        name="Duration",
        marker_color="steelblue",
        line_color="steelblue",
        boxpoints="outliers",   # muestra outliers como puntos
        showlegend=False,
        boxmean="sd"
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Age patient distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Age (years)", row=2, col=1)

fig.show()
fig.write_image("assets/age_patient_distribution.png", scale=5)

#### 3.3.1. Age distribution (by fragment)

In [6]:
# --- 1. Datos ---
ages = df["age"]

# --- 2. Estimar KDE ---
kde = gaussian_kde(ages)
x_kde = np.linspace(ages.min(), ages.max(), 300)
y_kde = kde(x_kde)

# --- 3. Crear figura: 2 filas, eje X compartido ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],   # histograma más grande, boxplot como "resumen" fino
    vertical_spacing=0.03,
)

# --- Histograma (normalizado a densidad para que la escala coincida con la KDE) ---
fig.add_trace(
    go.Histogram(
        x=ages,
        histnorm="probability density",
        name="Duration",
        marker_color="steelblue",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

# --- KDE ---
fig.add_trace(
    go.Scatter(
        x=x_kde,
        y=y_kde,
        mode="lines",
        name="KDE",
        line=dict(color="cadetblue", width=2.5),
        showlegend=False,
    ),
    row=1, col=1
)

# --- Boxplot horizontal ---
fig.add_trace(
    go.Box(
        x=ages,
        name="Duration",
        marker_color="steelblue",
        line_color="steelblue",
        boxpoints="outliers",   # muestra outliers como puntos
        showlegend=False,
        boxmean="sd"
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Age fragment distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Age (years)", row=2, col=1)

fig.show()
fig.write_image("assets/filtered/age_fragment_distribution.png", scale=5)

In [7]:
ages.describe()

count    20871.000000
mean         5.136591
std          3.136835
min          0.000000
25%          3.200000
50%          4.600000
75%          6.900000
max         55.000000
Name: age, dtype: float64

#### 3.3.2. Age by class

In [8]:
# --- 3. Crear figura: 2 filas, eje X compartido ---
fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    row_heights=[0.90, 0.25, 0.25, 0.25, 0.25],   # histograma más grande, boxplot como "resumen" fino
    vertical_spacing=0.03,
)


# --- 1. Datos ---
labels = ["Normal", "Wheeze", "Rhonchi", "Stridor"]

for i, label in enumerate(labels, start=2):

    df_i = df[df["label"] == label]
    ages = df_i["age"].dropna()

    # --- 2. Estimar KDE ---
    kde = gaussian_kde(ages)
    x_kde = np.linspace(ages.min(), ages.max(), 300)
    y_kde = kde(x_kde)

    # --- Histograma (normalizado a densidad para que la escala coincida con la KDE) ---
    fig.add_trace(
        go.Histogram(
            x=ages,
            histnorm="probability density",
            name="Duration",
            marker_color=color_map[label],
            opacity=0.4,
            showlegend=False,
        ),
        row=1, col=1
    )

    # --- KDE ---
    fig.add_trace(
        go.Scatter(
            x=x_kde,
            y=y_kde,
            mode="lines",
            name="KDE",
            line=dict(color=color_map[label], width=2.5),
            showlegend=False,
        ),
        row=1, col=1
    )

    # --- Boxplot horizontal ---
    fig.add_trace(
        go.Box(
            x=ages,
            name=label,
            marker_color=color_map[label],
            line_color=color_map[label],
            boxpoints="outliers",   # muestra outliers como puntos
            showlegend=False,
            boxmean="sd"
        ),
        row=i, col=1
    )

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Age distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
    barmode="overlay",
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_xaxes(title_text="Age (years)", row=8, col=1)

fig.show()
fig.write_image("assets/filtered/age_distribution_class.png", scale=5)

### 3.4. Duration distribution

In [9]:
# --- 1. Datos ---
durations = df["duration"].dropna()

# --- 2. Estimar KDE ---
kde = gaussian_kde(durations)
x_kde = np.linspace(durations.min(), durations.max(), 300)
y_kde = kde(x_kde)

# --- 3. Crear figura: 2 filas, eje X compartido ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],   # histograma más grande, boxplot como "resumen" fino
    vertical_spacing=0.03,
)

# --- Histograma (normalizado a densidad para que la escala coincida con la KDE) ---
fig.add_trace(
    go.Histogram(
        x=durations,
        histnorm="probability density",
        name="Duration",
        marker_color="chocolate",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

# --- KDE ---
fig.add_trace(
    go.Scatter(
        x=x_kde,
        y=y_kde,
        mode="lines",
        name="KDE",
        line=dict(color="saddlebrown", width=2.5),
        showlegend=False,
    ),
    row=1, col=1
)

# --- Boxplot horizontal ---
fig.add_trace(
    go.Box(
        x=durations,
        name="Duration",
        marker_color="chocolate",
        line_color="chocolate",
        boxpoints="outliers",   # muestra outliers como puntos
        showlegend=False,
        boxmean="sd"
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Duration distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Duration (s)", row=2, col=1)

fig.show()
fig.write_image("assets/filtered/duration_distribution.png", scale=5)

In [10]:
df["duration"].describe()

count    20871.000000
mean         1.768341
std          0.762698
min          0.126250
25%          1.240250
50%          1.705250
75%          2.218250
max          9.273250
Name: duration, dtype: float64

In [21]:
print(np.percentile(df["duration"], [90, 95, 98, 99]))

[2.72325 3.07175 3.58345 3.97355]


In [28]:
np.mean(df["duration"] <= 4)

np.float64(0.9904652388481625)

### 3.4.1. Duration by class

In [11]:
# --- 3. Crear figura: 2 filas, eje X compartido ---
fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    row_heights=[0.9, 0.25, 0.25, 0.25, 0.25],   # histograma más grande, boxplot como "resumen" fino
    vertical_spacing=0.03,
)


# --- 1. Datos ---
labels = ["Normal", "Wheeze", "Rhonchi", "Stridor"]

for i, label in enumerate(labels, start=2):

    df_i = df[df["label"] == label]
    durations = df_i["duration"].dropna()

    # --- 2. Estimar KDE ---
    kde = gaussian_kde(durations)
    x_kde = np.linspace(durations.min(), durations.max(), 300)
    y_kde = kde(x_kde)

    # --- Histograma (normalizado a densidad para que la escala coincida con la KDE) ---
    fig.add_trace(
        go.Histogram(
            x=durations,
            histnorm="probability density",
            name="Duration",
            marker_color=color_map[label],
            opacity=0.4,
            showlegend=False,
        ),
        row=1, col=1
    )

    # --- KDE ---
    fig.add_trace(
        go.Scatter(
            x=x_kde,
            y=y_kde,
            mode="lines",
            name="KDE",
            line=dict(color=color_map[label], width=2.5),
            showlegend=False,
        ),
        row=1, col=1
    )

    # --- Boxplot horizontal ---
    fig.add_trace(
        go.Box(
            x=durations,
            name=label,
            marker_color=color_map[label],
            line_color=color_map[label],
            boxpoints="outliers",   # muestra outliers como puntos
            showlegend=False,
            boxmean="sd"
        ),
        row=i, col=1
    )

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Duration distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
    barmode="overlay"
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_xaxes(title_text="Duration (s)", row=8, col=1)

fig.show()
fig.write_image("assets/filtered/duration_distribution_class.png", scale=5)

### 3.5. Number of fragments per record

In [12]:
# --- 1. Contar fragmentos por paciente ---
fragments_per_record = df.groupby("record_id").size()

# --- 3. Figura: histograma + KDE arriba, boxplot horizontal abajo ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.03,
)

fig.add_trace(
    go.Histogram(
        x=fragments_per_record,
        name="n/patient",
        marker_color="mediumpurple",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Box(
        x=fragments_per_record,
        name="Fragments/patient",
        marker_color="mediumpurple",
        line_color="mediumpurple",
        boxpoints="outliers",
        showlegend=False,
        boxmean="sd",
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Number of fragments per record distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Number of fragments per record", row=2, col=1)

fig.show()
fig.write_image("assets/filtered/fragments_record_distribution.png", scale=5)

In [13]:
fragments_per_record.describe()

count    5480.000000
mean        3.808577
std         2.216276
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        24.000000
dtype: float64

### 3.6. Number of records per patient

In [14]:
# --- 1. Contar fragmentos por paciente ---
records_per_patient = df_all.groupby(["patient_id", "age"]).size()

# --- 2. Estimar KDE (si hay suficiente variación en los valores) ---
kde = gaussian_kde(records_per_patient)
x_kde = np.linspace(records_per_patient.min(), records_per_patient.max(), 300)
y_kde = kde(x_kde)

# --- 3. Figura: histograma + KDE arriba, boxplot horizontal abajo ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.03,
)

fig.add_trace(
    go.Histogram(
        x=records_per_patient,
        histnorm="probability density",
        name="n/patient",
        marker_color="#aee394",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=x_kde,
        y=y_kde,
        mode="lines",
        name="KDE",
        line=dict(color="#669932", width=2.5),
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Box(
        x=records_per_patient,
        name="Records/patient",
        marker_color="#aee394",
        line_color="#aee394",
        boxpoints="outliers",
        showlegend=False,
        boxmean="sd",
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Number of records per patient distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Number of records per patient", row=2, col=1)

fig.show()
fig.write_image("assets/filtered/records_patient_distribution.png", scale=5)

In [15]:
records_per_patient.describe()

count    905.000000
mean       6.825414
std        7.063117
min        1.000000
25%        3.000000
50%        5.000000
75%        8.000000
max       83.000000
dtype: float64

### 3.7. Number of fragments per patient

In [16]:
# --- 1. Contar fragmentos por paciente ---
fragments_per_patient = df.groupby("name").size()

# --- 2. Estimar KDE (si hay suficiente variación en los valores) ---
kde = gaussian_kde(fragments_per_patient)
x_kde = np.linspace(fragments_per_patient.min(), fragments_per_patient.max(), 300)
y_kde = kde(x_kde)

# --- 3. Figura: histograma + KDE arriba, boxplot horizontal abajo ---
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.03,
)

fig.add_trace(
    go.Histogram(
        x=fragments_per_patient,
        histnorm="probability density",
        name="n/patient",
        marker_color="mediumpurple",
        opacity=0.75,
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=x_kde,
        y=y_kde,
        mode="lines",
        name="KDE",
        line=dict(color="rebeccapurple", width=2.5),
        showlegend=False,
    ),
    row=1, col=1
)

fig.add_trace(
    go.Box(
        x=fragments_per_patient,
        name="Fragments/patient",
        marker_color="mediumpurple",
        line_color="mediumpurple",
        boxpoints="outliers",
        showlegend=False,
        boxmean="sd",
    ),
    row=2, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Number of fragments per patient distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_xaxes(title_text="Number of fragments per patient", row=2, col=1)

fig.show()
fig.write_image("assets/filtered/fragments_patient_distribution.png", scale=5)

In [17]:
fragments_per_patient.describe()

count    920.00000
mean      22.68587
std       26.24585
min        1.00000
25%        7.00000
50%       15.00000
75%       29.00000
max      295.00000
dtype: float64

### 3.8. Labels per patient distribution

In [18]:
# --- 1. Número de labels distintos por paciente ---
labels_per_patient = df.groupby("name")["label"].nunique()

# --- 3. Figura: histograma + KDE arriba, boxplot horizontal abajo ---
fig = make_subplots(
    rows=1, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
)

fig.add_trace(
    go.Histogram(
        x=labels_per_patient,
        name="Labels/patient",
        marker_color="indianred",
        opacity=0.75,
        xbins=dict(size=1),
        showlegend=False,
    ),
    row=1, col=1
)

# --- 4. Layout ---
fig.update_layout(
    template="plotly_white",
    title_text="Number of labels per patient distribution",
    font=dict(size=15),
    title_font=dict(size=20),
    bargap=0.02,
)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_xaxes(title_text="Number of different labels", row=2, col=1)

fig.show()
fig.write_image("assets/filtered/labels_patient_distribution.png", scale=5)

In [19]:
labels_per_patient.describe()

count    920.000000
mean       1.185870
std        0.429105
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        3.000000
Name: label, dtype: float64

### 3.9. Age vs duration vs class

In [20]:
fig = go.Figure()
for label in labels_order:
    sub = df[df["label"] == label]

    fig.add_trace(
        go.Scatter(
            x=sub["age"], y=sub["duration"],
            mode="markers", name=label,
            marker=dict(
                color=color_map[label],
                size=6,
                opacity=0.7,
            )
        )
    )

fig.update_layout(
    template="plotly_white",
    title_text="Age vs Duration by label",
    font=dict(size=15),
    title_font=dict(size=20),
    legend=dict(traceorder="normal", font=dict(size=13)),
)
fig.update_xaxes(title_text="Age (years)")
fig.update_yaxes(title_text="Duration (s)")

fig.show()
fig.write_image("assets/filtered/age_vs_duration_vs_label.png", scale=5)